# Demo Modul 5: Convolutional Neural Network Dasar

**Durasi sesi:** 120 menit
**Fokus:** hitungan convolution yang dikerjakan dengan tangan, lalu dicocokkan dengan PyTorch.

Notebook ini mengikuti urutan Bagian A–E pada modul. Setiap sel kode didahului pertanyaan yang dijawab lebih dahulu di papan tulis; kode hanya dipakai untuk memverifikasi hitungan tersebut.

## Alur sesi

| Menit | Bagian |
|---|---|
| 20–40 | A. Cross-correlation manual |
| 40–55 | B. Padding, stride, dan banyak kanal |
| 55–70 | C. Unfold dan biaya komputasi |
| 70–90 | D. Gradien manual lawan autograd |
| 90–115 | E. Dari operasi ke jaringan |

In [ ]:
import platform
import random
import statistics
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10, FashionMNIST

DATASET = 'cifar10'      # ganti ke 'fashion' bila hanya tersedia CPU
NIM = '42'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DT = torch.float64

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cek(nama, hasil, acuan):
    """Sel pemeriksaan: shape harus sama dan nilai lolos torch.allclose."""
    assert hasil is not None and acuan is not None, f'{nama}: TODO belum diisi'
    hasil = torch.as_tensor(hasil.detach() if torch.is_tensor(hasil) else hasil, dtype=DT)
    acuan = torch.as_tensor(acuan.detach() if torch.is_tensor(acuan) else acuan, dtype=DT)
    ok = hasil.shape == acuan.shape and torch.allclose(hasil, acuan)
    print(f'[{"OK" if ok else "GAGAL"}] {nama}: shape {tuple(hasil.shape)}')
    assert ok, f'{nama} tidak cocok dengan acuan'

def ukur(fungsi, ulang=10):
    """Median waktu eksekusi `fungsi` dalam detik setelah satu pemanasan."""
    fungsi()
    waktu = []
    for _ in range(ulang):
        mulai = time.perf_counter()
        fungsi()
        waktu.append(time.perf_counter() - mulai)
    return statistics.median(waktu)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'python': platform.python_version(), 'torch': torch.__version__,
       'device': str(DEVICE), 'dataset': DATASET, 'seed': SEED})

## A. Cross-correlation manual

$$\mathbf{X}=\begin{bmatrix}1&2&0&1&3\\0&1&3&2&1\\2&1&0&1&2\\1&0&2&3&0\\3&1&1&0&2\end{bmatrix},\qquad
\mathbf{K}=\begin{bmatrix}1&0&-1\\2&1&0\\0&-1&1\end{bmatrix}$$

**Tanya kelas:** berapa $Y_{0,0}$ dan $Y_{0,1}$? Setelah dijawab, jalankan sel berikut.

In [ ]:
X = torch.tensor([[1, 2, 0, 1, 3],
                  [0, 1, 3, 2, 1],
                  [2, 1, 0, 1, 2],
                  [1, 0, 2, 3, 0],
                  [3, 1, 1, 0, 2]], dtype=DT)
K = torch.tensor([[1, 0, -1],
                  [2, 1, 0],
                  [0, -1, 1]], dtype=DT)

def corr2d(X, K):
    """Cross-correlation 2D satu kanal, tanpa padding, stride 1."""
    h, w = K.shape
    Y = torch.zeros(X.shape[0] - h + 1, X.shape[1] - w + 1, dtype=X.dtype)
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

Y_TANGAN = [[1, 7, 6], [4, 2, 0], [4, 1, 7]]

Y_manual = corr2d(X, K)
print('X ⋆ K =\n', Y_manual)
cek('hitungan tangan vs corr2d', Y_manual, Y_TANGAN)
cek('corr2d vs F.conv2d', Y_manual, F.conv2d(X[None, None], K[None, None])[0, 0])

Y_konvolusi = corr2d(X, torch.flip(K, (0, 1)))
print('X * K (kernel dibalik) =\n', Y_konvolusi)
assert not torch.allclose(Y_konvolusi, Y_manual), 'konvolusi matematis seharusnya berbeda'

## B. Padding, stride, dan banyak kanal — bagian dari 20 poin

$$Y_{c',i,j}=b_{c'}+\sum_{c}\sum_{u=0}^{k-1}\sum_{v=0}^{k-1}W_{c',c,u,v}\,\tilde{X}_{c,\,si+u,\,sj+v},\qquad
n_\text{keluar}=\left\lfloor\frac{n+2p-d(k-1)-1}{s}\right\rfloor+1$$

In [ ]:
def shape_keluar(n, k, p=0, s=1, d=1):
    """Rumus shape satu sumbu spasial, termasuk dilation."""
    return (n + 2 * p - d * (k - 1) - 1) // s + 1

def corr2d_multi(X, W, b=None, stride=1, padding=0):
    """X: (C_in, H, W), W: (C_out, C_in, k, k), b: (C_out,) -> (C_out, H_out, W_out)."""
    c_out, c_in, kh, kw = W.shape
    assert X.shape[0] == c_in, 'kanal masukan tidak cocok dengan kernel'
    Xp = F.pad(X, (padding, padding, padding, padding))
    h_out = shape_keluar(X.shape[1], kh, padding, stride)
    w_out = shape_keluar(X.shape[2], kw, padding, stride)
    Y = torch.zeros(c_out, h_out, w_out, dtype=X.dtype)
    for i in range(h_out):
        for j in range(w_out):
            r, c = i * stride, j * stride
            jendela = Xp[:, r:r + kh, c:c + kw]              # (C_in, k, k)
            Y[:, i, j] = (W * jendela).sum(dim=(1, 2, 3))    # satu nilai per kanal keluaran
    if b is not None:
        Y = Y + b.view(-1, 1, 1)
    return Y

PRELAB_A = [[2, 3, 4], [0, 2, 6], [3, 0, 5]]

Y_s2 = corr2d_multi(X[None], K[None, None], stride=2, padding=1)[0]
print('p=1, s=2:\n', Y_s2)
cek('pre-lab a vs corr2d_multi', Y_s2, PRELAB_A)
cek('corr2d_multi vs F.conv2d (p=1, s=2)', Y_s2,
    F.conv2d(X[None, None], K[None, None], stride=2, padding=1)[0, 0])

Xm = torch.tensor([[[1, 0, 2], [3, 1, 0], [0, 2, 1]],
                   [[2, 1, 0], [0, 1, 3], [1, 0, 2]]], dtype=DT)
Wm = torch.tensor([[[[1, -1], [0, 2]], [[0, 1], [-1, 1]]],
                   [[[1, 0], [0, 1]], [[1, 1], [1, 1]]]], dtype=DT)
bm = torch.tensor([0.5, -1.0], dtype=DT)
Ym_modul = torch.tensor([[[5.5, 0.5], [6.5, 8.5]],
                         [[5.0, 4.0], [6.0, 7.0]]], dtype=DT)

Ym = corr2d_multi(Xm, Wm, bm)
cek('contoh dua kanal vs modul', Ym, Ym_modul)

conv = nn.Conv2d(2, 2, 2).double()

with torch.no_grad():
    conv.weight.copy_(Wm)
    conv.bias.copy_(bm)

cek('contoh dua kanal vs nn.Conv2d', conv(Xm[None])[0], Ym)
print('parameter nn.Conv2d(2, 2, 2):', sum(q.numel() for q in conv.parameters()))

In [ ]:
KONFIGURASI = [  # (C_in, C_out, n, k, s, p)
    (3, 4, 7, 3, 1, 1),
    (3, 4, 9, 5, 2, 2),
    (1, 2, 6, 3, 2, 0),
    (2, 3, 8, 5, 1, 0),
    (3, 4, 11, 3, 2, 1),
]
gen = torch.Generator().manual_seed(SEED)
baris = []
for c_in, c_out, n, k, s, p in KONFIGURASI:
    Xr = torch.randn(c_in, n, n, dtype=DT, generator=gen)
    Wr = torch.randn(c_out, c_in, k, k, dtype=DT, generator=gen)
    br = torch.randn(c_out, dtype=DT, generator=gen)
    Y_manual = corr2d_multi(Xr, Wr, br, stride=s, padding=p)
    Y_acuan = F.conv2d(Xr[None], Wr, br, stride=s, padding=p)[0]
    shape_rumus = (c_out, shape_keluar(n, k, p, s), shape_keluar(n, k, p, s))
    baris.append({'C_in': c_in, 'C_out': c_out, 'n': n, 'k': k, 's': s, 'p': p,
                  'shape_rumus': shape_rumus, 'shape_manual': tuple(Y_manual.shape),
                  'shape_cocok': shape_rumus == tuple(Y_acuan.shape) == tuple(Y_manual.shape),
                  'allclose': torch.allclose(Y_manual, Y_acuan)})
tabel_b = pd.DataFrame(baris)
display(tabel_b)
assert tabel_b['shape_cocok'].all() and tabel_b['allclose'].all(), 'ada konfigurasi yang belum lolos'
print('kelima konfigurasi lolos')

## C. Unfold dan biaya komputasi — 10 poin

$$\operatorname{vec}(\mathbf{Y})=\mathbf{W}_\text{flat}\,\mathbf{U}+\mathbf{b}\mathbf{1}^\top,\qquad
\mathbf{U}\in\mathbb{R}^{C_\text{in}k^2\times L},\qquad
\text{perkalian}=C_\text{out}C_\text{in}k^2H_\text{keluar}W_\text{keluar}$$

In [ ]:
PRELAB_B = [1, 3, 2, 1, 0, 1, 0, 2, 3]

U = F.unfold(X[None, None], 3)
Y_unfold = (K.reshape(1, -1) @ U).view(3, 3)

print('shape U:', tuple(U.shape))
assert tuple(U.shape) == (1, 9, 9)
cek('kolom ke-4 U vs pre-lab b', U[0, :, 4], PRELAB_B)
cek('unfold vs corr2d', Y_unfold, corr2d(X, K))

def conv_unfold(X, W, b=None, stride=1, padding=0):
    """X: (C_in, H, W) -> (C_out, H_out, W_out) memakai F.unfold dan perkalian matriks."""
    c_out, c_in, kh, kw = W.shape
    h_out = shape_keluar(X.shape[1], kh, padding, stride)
    w_out = shape_keluar(X.shape[2], kw, padding, stride)
    U = F.unfold(X[None], (kh, kw), padding=padding, stride=stride)[0]   # (C_in*k*k, L)
    Y = W.reshape(c_out, -1) @ U                                         # (C_out, L)
    if b is not None:
        Y = Y + b.view(-1, 1)
    return Y.view(c_out, h_out, w_out)

c_in, c_out, n, k, s, p = KONFIGURASI[1]
Xr = torch.randn(c_in, n, n, dtype=DT, generator=gen)
Wr = torch.randn(c_out, c_in, k, k, dtype=DT, generator=gen)
br = torch.randn(c_out, dtype=DT, generator=gen)
cek('conv_unfold vs F.conv2d', conv_unfold(Xr, Wr, br, s, p),
    F.conv2d(Xr[None], Wr, br, stride=s, padding=p)[0])

In [ ]:
def matriks_konvolusi(K, n):
    """Bangun M sehingga vec(corr2d(X, K)) = M @ vec(X) untuk X berukuran n x n."""
    kolom = []
    for q in range(n * n):
        e = torch.zeros(n * n, dtype=K.dtype)
        e[q] = 1.0
        kolom.append(corr2d(e.view(n, n), K).flatten())
    return torch.stack(kolom, dim=1)

M = matriks_konvolusi(K, 5)
print('shape M:', tuple(M.shape), '| entri tak nol:', int((M != 0).sum()), 'dari', M.numel())
cek('M @ vec(X) vs vec(X ⋆ K)', M @ X.flatten(), corr2d(X, K).flatten())
assert all(torch.equal(M[r][M[r] != 0], K[K != 0]) for r in range(M.shape[0])), \
    'setiap baris M seharusnya memakai bobot yang sama (parameter sharing)'
print('setiap baris M memakai bilangan yang sama dengan K')

In [ ]:
torch.manual_seed(SEED)
Xb = torch.randn(8, 3, 32, 32)        # float32 cukup untuk pengukuran waktu
Wb = torch.randn(16, 3, 3, 3)
bb = torch.randn(16)

implementasi = {
    'corr2d_multi (loop)': lambda: torch.stack([corr2d_multi(x, Wb, bb, padding=1) for x in Xb]),
    'conv_unfold (matmul)': lambda: torch.stack([conv_unfold(x, Wb, bb, padding=1) for x in Xb]),
    'F.conv2d': lambda: F.conv2d(Xb, Wb, bb, padding=1),
}

rujukan = F.conv2d(Xb, Wb, bb, padding=1)
baris = []
for nama, fungsi in implementasi.items():
    assert fungsi is not None, f'{nama}: TODO belum diisi'
    assert torch.allclose(fungsi(), rujukan, atol=1e-4), f'{nama} tidak cocok dengan F.conv2d'
    baris.append({'implementasi': nama, 'median_detik': ukur(fungsi)})
tabel_waktu = pd.DataFrame(baris)
tabel_waktu['kali_lebih_lambat'] = tabel_waktu['median_detik'] / tabel_waktu['median_detik'].iloc[-1]
display(tabel_waktu)

## D. Gradien manual lawan autograd

$$\mathbf{X}=\begin{bmatrix}1&2&0\\0&1&3\\2&1&1\end{bmatrix},\quad
\mathbf{K}=\begin{bmatrix}1&-1\\2&0\end{bmatrix},\quad
\mathbf{G}=\frac{\partial\mathcal{L}}{\partial\mathbf{Y}}=\begin{bmatrix}1&0\\-1&2\end{bmatrix},\qquad
\frac{\partial\mathcal{L}}{\partial\mathbf{K}}=\mathbf{X}\star\mathbf{G},\;\;
\frac{\partial\mathcal{L}}{\partial b}=\sum G_{i,j},\;\;
\frac{\partial\mathcal{L}}{\partial\mathbf{X}}=\operatorname{pad}_{k-1}(\mathbf{G})\star\operatorname{rot180}(\mathbf{K})$$

**Tanya kelas:** piksel $X_{1,1}$ dipakai oleh posisi keluaran mana saja, dan dengan bobot kernel yang mana?

In [ ]:
Xg = torch.tensor([[1, 2, 0], [0, 1, 3], [2, 1, 1]], dtype=DT)
Kg = torch.tensor([[1, -1], [2, 0]], dtype=DT)
G = torch.tensor([[1, 0], [-1, 2]], dtype=DT)

DK_TANGAN = [[3, 7], [0, 2]]
DB_TANGAN = [2]
DX_TANGAN = [[1, -1, 0], [1, 3, -2], [-2, 4, 0]]

def gradien_manual(X, K, G):
    """Gradien cross-correlation satu kanal untuk stride 1 tanpa padding."""
    k = K.shape[0]
    dK = corr2d(X, G)
    db = G.sum().reshape(1)
    dX = corr2d(F.pad(G, (k - 1, k - 1, k - 1, k - 1)), torch.flip(K, (0, 1)))
    return dK, db, dX

dK, db, dX = gradien_manual(Xg, Kg, G)

X_ag = Xg.clone().requires_grad_(True)
K_ag = Kg.clone().requires_grad_(True)
b_ag = torch.zeros(1, dtype=DT, requires_grad=True)

Y_ag = F.conv2d(X_ag[None, None], K_ag[None, None], b_ag)[0, 0]
(G * Y_ag).sum().backward()

for nama, manual, tangan, acuan in [('dL/dK', dK, DK_TANGAN, K_ag.grad),
                                    ('dL/db', db, DB_TANGAN, b_ag.grad),
                                    ('dL/dX', dX, DX_TANGAN, X_ag.grad)]:
    cek(f'{nama} tangan vs manual', manual, tangan)
    cek(f'{nama} manual vs autograd', manual, acuan)

In [ ]:
P = torch.tensor([[1, 3, 2, 0], [4, 2, 1, 5], [0, 1, 3, 2], [2, 6, 0, 1]], dtype=DT)
G_pool = torch.tensor([[1, 2], [3, 4]], dtype=DT)

def maxpool_manual(P, G_pool, k=2):
    """Max pooling k x k stride k beserta gradiennya."""
    h, w = P.shape[0] // k, P.shape[1] // k
    Z = torch.zeros(h, w, dtype=P.dtype)
    dP = torch.zeros_like(P)
    for i in range(h):
        for j in range(w):
            jendela = P[i * k:(i + 1) * k, j * k:(j + 1) * k]
            u, v = divmod(int(jendela.argmax()), k)
            Z[i, j] = jendela[u, v]
            dP[i * k + u, j * k + v] += G_pool[i, j]
    return Z, dP

Z, dP = maxpool_manual(P, G_pool)
P_ag = P.clone().requires_grad_(True)
Z_ag = F.max_pool2d(P_ag[None, None], 2)[0, 0]
(G_pool * Z_ag).sum().backward()
cek('max pooling manual vs F.max_pool2d', Z, Z_ag)
cek('gradien max pooling vs autograd', dP, P_ag.grad)
print('dL/dP =\n', dP)

## E. Dari operasi ke jaringan

**Tanya kelas:** isi kolom keluaran, parameter, perkalian, dan receptive field CNN baseline untuk masukan $3\times32\times32$ sebelum sel berikut dijalankan.

In [ ]:
def buat_cnn(c1=32, c2=64, k=3, c_in=3, hw=32):
    seed_everything(SEED)
    return nn.Sequential(
        nn.Conv2d(c_in, c1, kernel_size=k, padding=k // 2), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(c1, c2, kernel_size=k, padding=k // 2), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(c2 * (hw // 4) ** 2, 128), nn.ReLU(),
        nn.Linear(128, 10),
    )

def buat_fnn(hidden=177, c_in=3, hw=32):
    seed_everything(SEED)
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(c_in * hw * hw, hidden), nn.ReLU(),
        nn.Linear(hidden, 10),
    )

def anatomi(model, shape_masukan):
    """Tabel shape, parameter, perkalian, dan receptive field per child module."""
    baris, kait = [], []
    status = {'r': 1, 'j': 1, 'rata': False}

    def buat_kait(nama):
        def catat(modul, masukan, keluaran):
            perkalian = 0
            if isinstance(modul, nn.Conv2d):
                c_out, c_in_, kh, kw = modul.weight.shape
                perkalian = c_out * c_in_ * kh * kw * keluaran.shape[-2] * keluaran.shape[-1]
            elif isinstance(modul, nn.Linear):
                perkalian = modul.in_features * modul.out_features
            if isinstance(modul, (nn.Conv2d, nn.MaxPool2d)):
                k = modul.kernel_size if isinstance(modul.kernel_size, int) else modul.kernel_size[0]
                s = modul.stride if isinstance(modul.stride, int) else modul.stride[0]
                status['r'] += (k - 1) * status['j']
                status['j'] *= s
            status['rata'] = status['rata'] or isinstance(modul, nn.Flatten)
            baris.append({'layer': nama, 'modul': type(modul).__name__,
                          'keluaran': tuple(keluaran.shape[1:]),
                          'parameter': sum(q.numel() for q in modul.parameters()),
                          'perkalian': perkalian,
                          'rf': None if status['rata'] else status['r']})
        return catat

    for nama, modul in model.named_children():
        kait.append(modul.register_forward_hook(buat_kait(nama)))
    perangkat = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, *shape_masukan, device=perangkat))
    for h in kait:
        h.remove()
    tabel = pd.DataFrame(baris)
    tabel['rf'] = tabel['rf'].astype('Int64')
    return tabel

tabel_cnn = anatomi(buat_cnn(), (3, 32, 32))
tabel_fnn = anatomi(buat_fnn(), (3, 32, 32))
display(tabel_cnn)
display(tabel_fnn)

ringkas = pd.DataFrame({
    'model': ['CNN baseline', 'FNN pembanding'],
    'parameter': [int(tabel_cnn['parameter'].sum()), int(tabel_fnn['parameter'].sum())],
    'perkalian': [int(tabel_cnn['perkalian'].sum()), int(tabel_fnn['perkalian'].sum())],
})
display(ringkas)
assert ringkas['parameter'].tolist() == [545_098, 545_701], 'jumlah parameter belum sesuai modul'
assert ringkas['perkalian'].tolist() == [6_128_896, 545_514], 'jumlah perkalian belum sesuai modul'
assert int(tabel_cnn['rf'].dropna().iloc[-1]) == 10, 'receptive field keluaran blok kedua harus 10'

porsi_fc = tabel_cnn.loc[tabel_cnn['modul'] == 'Linear', 'parameter'].iloc[0] / ringkas.loc[0, 'parameter']
porsi_conv = tabel_cnn.loc[tabel_cnn['modul'] == 'Conv2d', 'perkalian'].sum() / ringkas.loc[0, 'perkalian']
rasio_perkalian = ringkas.loc[0, 'perkalian'] / ringkas.loc[1, 'perkalian']
print(f'parameter pada Linear pertama: {100 * porsi_fc:.1f}% | perkalian pada Conv2d: {100 * porsi_conv:.1f}%')
print(f'rasio perkalian CNN/FNN: {rasio_perkalian:.2f}')

In [ ]:
def rf_empiris(pool):
    seed_everything(SEED)
    blok = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), pool(2),
                         nn.Conv2d(32, 64, 3, padding=1), pool(2)).double()
    with torch.no_grad():
        for modul in blok:
            if isinstance(modul, nn.Conv2d):
                modul.weight.fill_(1.0)
                modul.bias.zero_()
    x = torch.rand(1, 3, 32, 32, dtype=DT, requires_grad=True)
    blok(x)[0, 0, 4, 4].backward()
    aktif = (x.grad[0].abs().sum(0) != 0).nonzero()
    tinggi = int(aktif[:, 0].max() - aktif[:, 0].min() + 1)
    lebar = int(aktif[:, 1].max() - aktif[:, 1].min() + 1)
    return tinggi, lebar

rf_avg, rf_max = rf_empiris(nn.AvgPool2d), rf_empiris(nn.MaxPool2d)
print('AvgPool2d:', rf_avg, '| MaxPool2d:', rf_max)
assert rf_avg == (10, 10), 'receptive field empiris dengan AvgPool2d harus 10 x 10'

In [ ]:
cnn_cpu, fnn_cpu = buat_cnn().eval(), buat_fnn().eval()
torch.manual_seed(SEED)
xb = torch.randn(128, 3, 32, 32)
with torch.no_grad():
    t_cnn = ukur(lambda: cnn_cpu(xb))
    t_fnn = ukur(lambda: fnn_cpu(xb))

print(f'forward CNN: {1e3 * t_cnn:.2f} ms | FNN: {1e3 * t_fnn:.2f} ms')
print(f'rasio waktu CNN/FNN: {t_cnn / t_fnn:.2f} | rasio perkalian: {rasio_perkalian:.2f}')

## Exit ticket

1. Hitung $Y_{1,1}$ dari $\mathbf{X}\star\mathbf{K}$ pada Bagian A tanpa melihat keluaran kode.
2. Mengapa CNN baseline memuat $96{,}2\%$ parameter pada `Linear`, tetapi $91{,}4\%$ perkaliannya pada `Conv2d`?
3. Mengapa gradien terhadap masukan memerlukan kernel yang dibalik?

**Tugas setelah sesi:** kerjakan `starter-mahasiswa.ipynb`, termasuk Tugas T1–T4.